In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp

# Setup Mediapipe
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True)
angle_points = {
    "left_elbow": (11, 13, 15),
    "right_elbow": (12, 14, 16),
    "left_knee": (23, 25, 27),
    "right_knee": (24, 26, 28),
    "left_shoulder": (13, 11, 23),
    "right_shoulder": (14, 12, 24),
    "hip": (11, 23, 25)
}

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    return 360 - angle if angle > 180 else angle

# Loop through dataset
data = []
base_path = "yoga_dataset"
for pose_name in os.listdir(base_path):
    folder = os.path.join(base_path, pose_name)
    if not os.path.isdir(folder): continue

    for file in os.listdir(folder):
        if not file.lower().endswith((".jpg", ".jpeg", ".png")): continue
        image_path = os.path.join(folder, file)
        image = cv2.imread(image_path)
        if image is None: continue

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)

        try:
            lm = results.pose_landmarks.landmark
            height, width = image.shape[:2]
            angles = []

            for p1, p2, p3 in angle_points.values():
                a = [lm[p1].x * width, lm[p1].y * height]
                b = [lm[p2].x * width, lm[p2].y * height]
                c = [lm[p3].x * width, lm[p3].y * height]
                angles.append(calculate_angle(a, b, c))

            angles.append(pose_name)
            data.append(angles)
        except:
            continue

# Save angle data
columns = list(angle_points.keys()) + ['label']
df = pd.DataFrame(data, columns=columns)
df.to_csv("pose_angles_dataset.csv", index=False)
print("Pose angle dataset saved.")



Pose angle dataset saved.


In [2]:
import pandas as pd

# Load the joint angle dataset
df = pd.read_csv("pose_angles_dataset.csv")  # Ensure this file has joint angles + 'label'

# Group by pose label and calculate mean of each angle
mean_angles = df.groupby("label").mean()

# Save the result to a CSV
mean_angles.to_csv("pose_mean_angles.csv")

print("Mean joint angles per pose saved to 'pose_mean_angles.csv'")

Mean joint angles per pose saved to 'pose_mean_angles.csv'


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Load data
df = pd.read_csv("real_pose_angle.csv")
X = df.drop("label", axis=1).values
y = LabelEncoder().fit_transform(df["label"])

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(len(np.unique(y)), activation='softmax'))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=30, validation_data=(X_test, y_test))

# Save model
model.save("angle_pose_model.h5")


c:\Users\shres\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.0000e+00 - loss: 32.9527 - val_accuracy: 0.0000e+00 - val_loss: 27.6804
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 220ms/step - accuracy: 0.2000 - loss: 25.6178 - val_accuracy: 0.0000e+00 - val_loss: 27.8025
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.2000 - loss: 19.3645 - val_accuracy: 0.0000e+00 - val_loss: 29.9726
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.2000 - loss: 12.7420 - val_accuracy: 0.0000e+00 - val_loss: 33.4961
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.0000e+00 - loss: 9.3312 - val_accuracy: 0.0000e+00 - val_loss: 36.4739
Epoch 6/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.4000 - loss: 9.3370 - val_accuracy: 0.0000e+00 - val_loss: 38.3064
Epoch 7/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.4000 - loss: 8.5530 - val_accuracy: 0.0000e+00 - val_loss: 39.9125
Epoch 8/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.4000 - los

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf

model = tf.keras.models.load_model("angle_pose_model.h5")
labels = df["label"].unique()

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    return 360 - angle if angle > 180 else angle

angle_points = {
    "left_elbow": (11, 13, 15),
    "right_elbow": (12, 14, 16),
    "left_knee": (23, 25, 27),
    "right_knee": (24, 26, 28),
    "left_shoulder": (13, 11, 23),
    "right_shoulder": (14, 12, 24),
    "hip": (11, 23, 25)
}


cap = cv2.VideoCapture(0)

with mp_pose.Pose(min_detection_confidence=0.6,
                  min_tracking_confidence=0.6) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert BGR to RGB for mediapipe
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(img_rgb)

        # Convert back to BGR for OpenCV visualization
        frame = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

        # Process landmarks
        if results.pose_landmarks:
            lm = results.pose_landmarks.landmark
            h, w = frame.shape[:2]

            # Collect angles
            angles = []
            all_angles_captured = True

            for p1, p2, p3 in angle_points.values():
                try:
                    a = [lm[p1].x * w, lm[p1].y * h]
                    b = [lm[p2].x * w, lm[p2].y * h]
                    c = [lm[p3].x * w, lm[p3].y * h]
                    angle = calculate_angle(a, b, c)
                    angles.append(angle)
                except:
                    all_angles_captured = False
                    break

            # Only predict if all angles were successfully calculated
            if all_angles_captured and len(angles) == len(angle_points):
                input_data = np.array(angles).reshape(1, -1)
                pred = model.predict(input_data)[0]
                index = np.argmax(pred)
                confidence = pred[index]

                if confidence > 0.9:
                    label = labels[index]
                    color = (0, 255, 0)  # green
                else:
                    label = "Unknown Pose"
                    color = (0, 0, 255)  # red

                # Display pose label
                cv2.putText(frame, f'{label} ({confidence*100:.1f}%)', (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

            # Draw landmarks
            mp_drawing.draw_landmarks(
                frame,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS
            )

        # Show frame
        cv2.imshow('Real-time Yoga Pose Detection', frame)

        # Break on 'q'
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━